# 01 - Run zero-shot baselines on FLORES-200
Runs on the Colab T4. Gets the latest code from GitHub, then runs `src/translate.py` with a config.

Edit code on your laptop -> push to GitHub -> re-run cell 2 here to pull the change.

In [28]:
# Cell 1: GPU check + Hugging Face login
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - switch runtime to T4")

from huggingface_hub import login, whoami
try:
    whoami()
    print("Already logged in to Hugging Face")
except Exception:
    from getpass import getpass
    login(token=getpass("Paste your Hugging Face token: "))

GPU: Tesla T4
Already logged in to Hugging Face


In [35]:
# Cell 2: get (or update) the code from GitHub and install packages
import os
REPO_URL = "https://github.com/prefierolasoledad/TourMT-Hi.git"
WORKDIR = "/content/tourmt-hi"

if not os.path.exists(WORKDIR):
    !git clone {REPO_URL} {WORKDIR}
else:
    # Make the Colab copy exactly match GitHub (GitHub is the master copy;
    # files changed only here on Colab are replaced by GitHub's version).
    !git -C {WORKDIR} fetch -q origin && git -C {WORKDIR} reset -q --hard origin/main
%cd {WORKDIR}
# Install packages; show only errors and the final summary (so failures are not hidden)
!pip install -r requirements.txt 2>&1 | grep -iE "error|failed|successfully installed" || true

# Check the packages we need actually import
import importlib
for pkg in ["transformers", "datasets", "sacrebleu"]:
    try:
        m = importlib.import_module(pkg)
        print(f"OK      {pkg} {getattr(m, '__version__', '')}")
    except Exception as e:
        print(f"MISSING {pkg}: {type(e).__name__}: {e}")
!git log -1 --oneline

/content/tourmt-hi
OK      transformers 5.22.1
OK      datasets 4.8.5
OK      sacrebleu 2.6.0
2979324 (HEAD -> main, origin/main, origin/HEAD) Update model configuration and translation logic for IndicTrans2 compatibility


In [36]:
# Cell 2b: separate Python 3.11 environment for evaluation + IndicTrans2 (both need transformers<5)
# Takes 1-2 minutes; needed once per Colab session. See requirements-eval.txt.
import os
if not os.path.exists("/content/venv-eval/bin/python"):
    !pip install -q uv
    !uv venv -q /content/venv-eval --python 3.11
# Always run: installs anything new in requirements-eval.txt (fast if nothing changed)
!uv pip install -q --python /content/venv-eval/bin/python -r requirements-eval.txt
# --no-deps on purpose: its dependency is a broken fork of indicnlp (see requirements-eval.txt)
!uv pip install -q --python /content/venv-eval/bin/python --no-deps IndicTransToolkit==1.1.1
!/content/venv-eval/bin/python -c "import sys; sys.path.insert(0, 'src'); import comet, sacrebleu, torch, transformers, datasets; from IndicTransToolkit.processor import IndicProcessor; from common import check_indicnlp_normalizer; check_indicnlp_normalizer(); from importlib.metadata import version; print('eval env OK | comet', version('unbabel-comet'), '| transformers', transformers.__version__, '| GPU', torch.cuda.is_available())"

/content/venv-eval/lib/python3.11/site-packages/torchmetrics/utilities/imports.py:23: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution
eval env OK | comet 2.2.7 | transformers 4.57.6 | GPU True


In [ ]:
# Cell 3: smoke test - 20 sentences only (about a minute)
!python src/translate.py --config configs/nllb600m_flores-devtest_en-hi.yaml --limit 20

In [ ]:
# Cell 4: full NLLB-600M runs on FLORES devtest (1012 sentences each)
for cfg in ["nllb600m_flores-devtest_en-hi", "nllb600m_flores-devtest_hi-en"]:
    !python src/translate.py --config configs/{cfg}.yaml

In [37]:
# Cell 4b: IndicTrans2-200M runs on FLORES devtest (uses the Python 3.11 environment from Cell 2b)
# The models are gated: accept the terms once on both pages while logged in to Hugging Face:
#   https://huggingface.co/ai4bharat/indictrans2-en-indic-dist-200M
#   https://huggingface.co/ai4bharat/indictrans2-indic-en-dist-200M
PY = "/content/venv-eval/bin/python"
!{PY} src/translate.py --config configs/it2-200m_flores-devtest_en-hi.yaml --limit 20   # quick test first
for cfg in ["it2-200m_flores-devtest_en-hi", "it2-200m_flores-devtest_hi-en"]:
    !{PY} src/translate.py --config configs/{cfg}.yaml





it2-200m_flores-devtest_en-hi: 20 sentences, en-hi, model ai4bharat/indictrans2-en-indic-dist-200M on cuda
/content/venv-eval/lib/python3.11/site-packages/triton/knobs.py:504: UserWarning: TRITON_DEFAULT_FP_FUSION is unset: defaulting to FP fusion ENABLED for backward compatibility in Triton 3.9. The default will change to DISABLED in Triton 3.10; set TRITON_DEFAULT_FP_FUSION=1 to keep fusion enabled or =0 to adopt the new default early and silence this warning.
  warn_once(FP_FUSION_DEFAULT_WARNING)
  20/20
Done in 9.6s
Saved to outputs/it2-200m_flores-devtest_en-hi_limit20

SRC: "We now have 4-month-old mice that are non-diabetic that used to be diabetic," he added.
HYP: उन्होंने कहा, "अब हमारे पास 4 महीने के चूहे हैं जो गैर-मधुमेह वाले हैं जो मधुमेह के रोगी हुआ करते थे।"
REF: उन्होंने कहा “कि अब हमारे पास 4 महीने उम्र वाले चूहे हैं जिन्हें मधुमेह नहीं है जो मधुमेह के रोगी थे। ”

SRC: Dr. Ehud Ur, professor of medicine at Dalhousie University in Halifax, Nova Scotia and chair of 

In [38]:
# Cell 5: score ALL FLORES runs (chrF++, BLEU, COMET, CometKiwi) -> results/results.csv
# Uses the separate evaluation environment from Cell 2b.
# CometKiwi is gated: accept its terms on https://huggingface.co/Unbabel/wmt22-cometkiwi-da first.
!/content/venv-eval/bin/python src/evaluate.py --runs $(ls -d outputs/*_flores-devtest_* | grep -v _limit)
!cut -d, -f1,7-10 results/results.csv | column -s, -t


== it2-200m_flores-devtest_en-hi (1012 sentences, normalisation: indicnlp-hi (nukta + alternative forms))
  chrF++ 58.89   BLEU 36.75
/content/venv-eval/lib/python3.11/site-packages/torchmetrics/utilities/imports.py:23: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution

Lightning automatically upgraded your loaded checkpoint from v1.8.3.post1 to v2.6.6. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint ../../root/.cache/huggingface/hub/models--Unbabel--wmt22-comet-da/snapshots/2760a223ac957f30acfb18c8aa649b01cf1d75f2/checkpoints/model.ckpt`
Encoder model frozen.
/content/venv-eval/lib/python3.11/site-packages/pytorch_lightning/core/saving.py:216: Found keys that are not i

In [39]:
# Cell 6: download the outputs to your laptop as one zip file
!cd {WORKDIR} && zip -qr /content/outputs.zip outputs results -x '*_limit*'
print("Created /content/outputs.zip - download it and unzip it into the project folder on your laptop")

Created /content/outputs.zip - download it and unzip it into the project folder on your laptop


In [40]:
# Cell 7: put the zip into this notebook's output so it can be rebuilt on the laptop
import base64
data = open("/content/outputs.zip", "rb").read()
print(f"zip size: {len(data)/1024:.0f} KB")
print("===ZIP-START===")
print(base64.b64encode(data).decode())
print("===ZIP-END===")

zip size: 799 KB
===ZIP-START===
...
===ZIP-END===
